# GameForge3D — Phase 4: Texture Generator (U-Net)

**FYP 2026-2027 | NUML Dept. of Computer Science**  
**Supervisor:** Ms. Tooba Sagheer

This notebook builds a **U-Net based Texture Generator** that produces  
UV-space texture maps (512×512 RGB) for game assets from category + prompt.

**Approach:** Fine-tune a U-Net on Stable Diffusion's VAE encoder/decoder  
to generate tileable, category-aware textures.

**Steps:**
1. Mount Google Drive
2. Install dependencies
3. Define U-Net architecture
4. Prepare texture training data (from Cap3D gaming subset)
5. Train U-Net texture model
6. Generate textures for each asset category
7. Apply texture to .obj mesh
8. Save checkpoint to Drive

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/GameForge3D/checkpoints/texture'
DATA_DIR       = '/content/drive/MyDrive/GameForge3D/data'
OUTPUT_DIR     = '/content/drive/MyDrive/GameForge3D/outputs/textures'
SHAPES_DIR     = '/content/drive/MyDrive/GameForge3D/outputs/shapes'

for d in [CHECKPOINT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')
print(f'Texture outputs: {OUTPUT_DIR}')

## Step 2 — Install Dependencies

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision
!pip install -q trimesh Pillow matplotlib opencv-python-headless
print('All packages ready.')

## Step 3 — Define U-Net Architecture

Custom U-Net with skip connections for 512×512 RGB texture generation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)

class TextureUNet(nn.Module):
    """
    U-Net: takes noise + category embedding → 512x512 RGB texture.
    Input : (B, 4, 512, 512) — 3 noise channels + 1 category channel
    Output: (B, 3, 512, 512) — RGB texture
    """
    def __init__(self, num_classes=4):
        super().__init__()
        # Category embedding projected to spatial map
        self.cat_embed = nn.Embedding(num_classes, 64)
        self.cat_proj  = nn.Linear(64, 512 * 512)

        # Encoder
        self.enc1 = DoubleConv(4, 64)   # 512→512
        self.enc2 = DoubleConv(64, 128)  # 256→256
        self.enc3 = DoubleConv(128, 256) # 128→128
        self.enc4 = DoubleConv(256, 512) # 64→64
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4   = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4  = DoubleConv(1024, 512)
        self.up3   = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3  = DoubleConv(512, 256)
        self.up2   = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2  = DoubleConv(256, 128)
        self.up1   = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1  = DoubleConv(128, 64)

        # Output head
        self.out = nn.Sequential(
            nn.Conv2d(64, 3, 1),
            nn.Sigmoid()  # RGB in [0,1]
        )

    def forward(self, noise, cat_id):
        # Category spatial map
        cat = self.cat_proj(self.cat_embed(cat_id))  # (B, 512*512)
        cat = cat.view(-1, 1, 512, 512)               # (B,1,512,512)
        x   = torch.cat([noise, cat], dim=1)          # (B,4,512,512)

        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b  = self.bottleneck(self.pool(e4))

        # Decoder with skip connections
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out(d1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = TextureUNet(num_classes=4).to(device)
total  = sum(p.numel() for p in model.parameters())
print(f'TextureUNet ready on {device}')
print(f'Parameters: {total:,}')

## Step 4 — Prepare Texture Training Data

We use Stable Diffusion to synthesize category-specific texture images  
as training targets for the U-Net. (No large dataset download needed.)

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from PIL import Image
import os

SD_MODEL = 'runwayml/stable-diffusion-v1-5'
print(f'Loading Stable Diffusion: {SD_MODEL}')

sd_pipe = StableDiffusionPipeline.from_pretrained(
    SD_MODEL,
    torch_dtype=torch.float16
).to(device)
sd_pipe.safety_checker = None   # disable for asset generation
print('SD pipeline loaded.')

# Texture prompts per category
TEXTURE_PROMPTS = {
    0: [ # Creature
        'dragon scales seamless texture, dark green, fantasy game asset, 4k',
        'wolf fur seamless texture, brown grey, game ready, pbr material',
        'zombie skin seamless texture, pale green, horror game, pbr'
    ],
    1: [ # Prop
        'old wood plank seamless texture, brown, game asset, pbr material',
        'rusty metal seamless texture, orange brown, game ready, pbr',
        'ancient stone seamless texture, grey, game asset, pbr'
    ],
    2: [ # Vehicle
        'car metal paint seamless texture, red, game ready, pbr material',
        'military camouflage seamless texture, green brown, game asset',
        'sci-fi metal panel seamless texture, grey blue, game ready, pbr'
    ],
    3: [ # Weapon
        'medieval steel blade seamless texture, silver, game asset, pbr',
        'enchanted sword texture, blue glowing runes, fantasy game, pbr',
        'worn leather grip seamless texture, brown, game ready, pbr'
    ]
}

LABEL_NAMES = {0: 'Creature', 1: 'Prop', 2: 'Vehicle', 3: 'Weapon'}
SYNTH_DIR   = '/content/synth_textures'
os.makedirs(SYNTH_DIR, exist_ok=True)

synth_data = []  # list of (img_path, label_id)

for label_id, prompts in TEXTURE_PROMPTS.items():
    cat_name = LABEL_NAMES[label_id]
    print(f'Generating textures for [{cat_name}]...')
    for i, prompt in enumerate(prompts):
        with torch.autocast('cuda'):
            img = sd_pipe(prompt, width=512, height=512,
                          num_inference_steps=20).images[0]
        path = f'{SYNTH_DIR}/{cat_name}_{i}.png'
        img.save(path)
        synth_data.append((path, label_id))
        print(f'  Saved: {path}')

print(f'\nTotal synthetic textures: {len(synth_data)}')

## Step 5 — Train U-Net Texture Model

> **~5-10 minutes on T4 GPU**

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

class TextureDataset(Dataset):
    def __init__(self, data, img_size=512):
        self.data = data
        self.tf   = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ])
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        path, label = self.data[idx]
        img  = Image.open(path).convert('RGB')
        return self.tf(img), torch.tensor(label, dtype=torch.long)

dataset    = TextureDataset(synth_data)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

optimizer  = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion  = nn.MSELoss()

EPOCHS     = 30
loss_log   = []

print('Training TextureUNet...')
model.train()

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for imgs, labels in dataloader:
        imgs, labels = imgs.to(device), labels.to(device)

        # Random noise input (3 channels)
        noise = torch.randn(imgs.size(0), 3, 512, 512, device=device)

        pred  = model(noise, labels)     # (B, 3, 512, 512)
        loss  = criterion(pred, imgs)    # reconstruct target texture

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg = epoch_loss / len(dataloader)
    loss_log.append(avg)
    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch [{epoch:3d}/{EPOCHS}] Loss: {avg:.6f}')

# Plot loss
plt.figure(figsize=(8, 3))
plt.plot(loss_log)
plt.title('TextureUNet Training Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/texture_loss.png', dpi=150)
plt.show()
print('Training complete!')

## Step 6 — Generate Textures for Each Asset Category

In [ ]:
model.eval()

LABEL_NAMES = {0: 'Creature', 1: 'Prop', 2: 'Vehicle', 3: 'Weapon'}
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

TEXTURE_PATHS = {}

with torch.no_grad():
    for label_id, cat_name in LABEL_NAMES.items():
        noise  = torch.randn(1, 3, 512, 512, device=device)
        cat_t  = torch.tensor([label_id], device=device)
        out    = model(noise, cat_t)[0]            # (3, 512, 512)

        # Convert to PIL
        img_np  = (out.permute(1, 2, 0).cpu().numpy() * 255).astype('uint8')
        img_pil = Image.fromarray(img_np)

        # Save
        tex_path = f'{OUTPUT_DIR}/{cat_name.lower()}_texture.png'
        img_pil.save(tex_path)
        TEXTURE_PATHS[cat_name] = tex_path

        # Plot
        axes[label_id].imshow(img_np)
        axes[label_id].set_title(cat_name)
        axes[label_id].axis('off')
        print(f'[{cat_name}] texture saved: {tex_path}')

plt.suptitle('Generated Textures per Category', fontsize=13)
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/generated_textures.png', dpi=150)
plt.show()

## Step 7 — Apply Texture to .obj Mesh

In [ ]:
import trimesh, glob, shutil

# Map category name to texture
CAT_MAP = {
    'weapon'  : 'Weapon',
    'vehicle' : 'Vehicle',
    'creature': 'Creature',
    'prop'    : 'Prop'
}

obj_files = glob.glob(f'{SHAPES_DIR}/*.obj')
print(f'Found {len(obj_files)} .obj files')

for obj_path in obj_files:
    fname    = os.path.basename(obj_path)
    category = fname.split('_')[0].lower()
    cat_name = CAT_MAP.get(category)

    if not cat_name:
        continue

    tex_path = TEXTURE_PATHS.get(cat_name)
    if not tex_path:
        continue

    # Load mesh
    mesh = trimesh.load(obj_path, force='mesh')

    # Assign texture as vertex colors (UV mapping approximation)
    tex_img = Image.open(tex_path).resize((256, 256))
    tex_np  = np.array(tex_img)

    # Sample color per vertex from texture
    verts_norm = (mesh.vertices - mesh.vertices.min(0)) / \
                 (mesh.vertices.max(0) - mesh.vertices.min(0) + 1e-8)
    u = (verts_norm[:, 0] * 255).astype(int).clip(0, 255)
    v = (verts_norm[:, 1] * 255).astype(int).clip(0, 255)
    colors = tex_np[v, u]  # (N, 3)
    alpha  = np.full((len(colors), 1), 255, dtype='uint8')
    mesh.visual.vertex_colors = np.hstack([colors, alpha])

    # Save textured mesh
    out_name = f'{OUTPUT_DIR}/textured_{category}.obj'
    mesh.export(out_name)
    print(f'Textured mesh saved: {out_name}')

print('All meshes textured!')

## Step 8 — Save Checkpoint to Google Drive

In [ ]:
import json

SAVE_PATH = f'{CHECKPOINT_DIR}/texture_unet_v1.pt'
torch.save({
    'model_state_dict' : model.state_dict(),
    'label_names'      : LABEL_NAMES,
    'img_size'         : 512,
    'num_classes'      : 4
}, SAVE_PATH)

size_mb = os.path.getsize(SAVE_PATH) / 1024 / 1024
print(f'Checkpoint saved: {SAVE_PATH}')
print(f'Size: {size_mb:.1f} MB')

## Phase 4 Complete! ✅

| Output | Location |
|--------|----------|
| U-Net Checkpoint | `checkpoints/texture/texture_unet_v1.pt` |
| Creature Texture | `outputs/textures/creature_texture.png` |
| Prop Texture | `outputs/textures/prop_texture.png` |
| Vehicle Texture | `outputs/textures/vehicle_texture.png` |
| Weapon Texture | `outputs/textures/weapon_texture.png` |
| Textured Meshes | `outputs/textures/textured_*.obj` |

**Pipeline so far:**
```
Text Prompt
    ↓  Phase 2: DistilBERT Router
Category
    ↓  Phase 3: Shap-E Generator
3D Mesh (.obj)
    ↓  Phase 4: U-Net Texture
Textured Mesh (.obj)
```

**Next → Phase 5: LOD Generation + STL Validation**